# Faruq-v3 AF2RN — seed-42 training — Kaggle

Attach the latest private dataset `faruq-v3-experiment-core-v1`, select a **T4 GPU**, enable **Internet**, then use **Run All**. This notebook runs the AF2RN static and all-1,665-train observability gates first. Only a double PASS starts the frozen 50-epoch seed-42 training from `D0_seed42_best.pt`.

Validation is opened only after the pre-training gates pass. Test is never available or accessed. `last.pt`, `best.pt`, log, reports, decision versus original AF2, and the run contract are included in the output ZIP.

In [ ]:
import json,os,time
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir(): raise RuntimeError('Notebook ini khusus Kaggle.')
os.chdir(WORK)
print('INDEXING /kaggle/input ONCE ...',flush=True)
started=time.time(); index={}; files=0
for root,dirs,names in os.walk(INPUT):
    for name in names:
        path=Path(root)/name; index.setdefault(name,[]).append(path); files+=1
print(f'INPUT INDEX READY: {files} files in {time.time()-started:.2f}s',flush=True)
def one(name):
    matches=sorted(index.get(name,[]))
    if len(matches)!=1: raise FileNotFoundError(f'STOP CEPAT: harus tepat satu {name}; ditemukan {matches}. Refresh/attach versi terbaru faruq-v3-experiment-core-v1.')
    return matches[0]
MANIFEST=one('af2_spectral_kaggle_manifest.json')
ARCHIVE=one('faruq-development-v3-grouped.tar.bin')
D0_INPUT=one('D0_seed42_best.pt')
AF2_RESULT_INPUT=one('lfdet_afab_seed42_screening.json')
manifest=json.loads(MANIFEST.read_text(encoding='utf-8'))
if manifest.get('format')!='coffee_detector.af2_spectral.kaggle_manifest.v2': raise RuntimeError('STOP CEPAT: manifest bukan spectral-v2; update private Kaggle Dataset dahulu.')
if manifest.get('test_images_included') is not False: raise RuntimeError('STOP: bundle mengekspos test.')
for name,path in [('faruq-development-v3-grouped.tar.bin',ARCHIVE),('D0_seed42_best.pt',D0_INPUT),('lfdet_afab_seed42_screening.json',AF2_RESULT_INPUT)]:
    contract=manifest.get('artifacts',{}).get(name,{})
    if path.stat().st_size!=int(contract.get('bytes',-1)): raise RuntimeError(f'STOP CEPAT: ukuran {name} tidak cocok manifest.')
proof=manifest.get('checkpoint_validation',{}).get('D0_seed42_best.pt',{})
if proof.get('loadable_by_ultralytics') is not True or int(proof.get('nc',-1))!=21: raise RuntimeError('STOP CEPAT: D0 seed-42 belum memiliki bukti load-test SNI-21.')
print('INPUT NAMES/SIZES PASS — belum clone/install/training')


In [ ]:
import importlib,importlib.metadata,shutil,subprocess,sys,torch
BRANCH='codex/af2-radially-normalized-angular-density'; REPO=WORK/'coffee-bean-detection'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(1,4):
    result=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],cwd=WORK)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==3: raise RuntimeError('git clone gagal tiga kali; cek Internet Kaggle.')
    time.sleep(2)
torch_before=importlib.metadata.version('torch')
subprocess.run([sys.executable,'-m','pip','install','-q','--disable-pip-version-check','ultralytics==8.4.96'],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True,cwd=WORK)
torch_after=importlib.metadata.version('torch')
if torch_after!=torch_before: raise RuntimeError(f'Instalasi mengubah Torch {torch_before} -> {torch_after}; restart session.')
for name in list(sys.modules):
    if name=='coffee_detector' or name.startswith('coffee_detector.'): sys.modules.pop(name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Kaggle sebelum Run All.')
GPU=torch.cuda.get_device_name(0); CAPABILITY=torch.cuda.get_device_capability(0)
if CAPABILITY[0]<7: raise RuntimeError(f'GPU {GPU} tidak kompatibel; pilih T4 lalu restart session.')
try: probe=torch.ones(1,device='cuda:0').sum().item()
except Exception as exc: raise RuntimeError(f'Kernel CUDA gagal pada {GPU}: {exc}') from exc
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('BRANCH:',BRANCH); print('COMMIT:',COMMIT); print('GPU:',GPU); print('TORCH:',torch_after); print('ULTRALYTICS:',__import__('ultralytics').__version__)


In [ ]:
from coffee_detector.experiments.prepare_af2rn_kaggle import prepare_af2rn_kaggle_input
from coffee_detector.af2_rn.audit import run_af2rn_static_audit
from coffee_detector.af2_rn.observability import run_af2rn_observability_audit
DATA,D0,CONTRACT=prepare_af2rn_kaggle_input(INPUT,WORK)
if CONTRACT.get('decision')!='PASS' or CONTRACT.get('validation_files_read') is not False or CONTRACT.get('test_images_accessed') is not False: raise RuntimeError('Kontrak input AF2RN gagal.')
AF2_RESULT=Path(CONTRACT['af2_result']); OUTPUT=WORK/'faruq-v3-af2rn-v1'; OUTPUT.mkdir(parents=True,exist_ok=True)
STATIC=OUTPUT/'static_audit.json'; OBS=OUTPUT/'observability_train.json'
(OUTPUT/'kaggle_input_contract.json').write_text(json.dumps(CONTRACT,indent=2)+'\n',encoding='utf-8')
static=run_af2rn_static_audit(D0,STATIC,device='cuda:0')
print('STATIC DECISION:',static['decision']); print('STATIC GATES:',static['gates'])
if static['decision']!='PASS': raise RuntimeError('STOP: static audit FAIL; training dilarang.')
observability=run_af2rn_observability_audit(DATA,DATA/'faruq_grouped_summary.json',STATIC,OBS,device='0',image_size=128,patches_per_image=16)
print('OBS DECISION:',observability['decision']); print('OBS GATES:',observability['gates']); print('DISTRIBUTIONS:',observability['distributions']); print('RADIAL:',observability['radial_retention'])
if observability['decision']!='PASS' or observability.get('training_authorized') is not True: raise RuntimeError('STOP: observability FAIL; training dilarang.')
print('DOUBLE PRE-TRAINING GATE PASS — AF2RN seed-42 training authorized')


In [ ]:
LOG=OUTPUT/'AF2RN_seed42_run.log'; RESULT=OUTPUT/'val_reports/AF2RN_seed42_result.json'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2rn','--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--d0-checkpoint',str(D0),'--static-audit',str(STATIC),'--observability-audit',str(OBS),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
if not RESULT.is_file():
    print('START/RESUME AF2RN | log=',LOG,flush=True)
    with LOG.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
    shown=None
    while process.poll() is None:
        csv_path=OUTPUT/'AF2RN/AF2RN_seed42/results.csv'
        epochs=max(0,len(csv_path.read_text(errors='replace').splitlines())-1) if csv_path.is_file() else 0
        if epochs!=shown: print(f'AF2RN: {epochs}/50 epoch tercatat | log={LOG}',flush=True); shown=epochs
        time.sleep(120)
    if process.returncode:
        tail='\n'.join(LOG.read_text(errors='replace').splitlines()[-180:]); print(tail,flush=True)
        raise RuntimeError(f'AF2RN gagal: {process.returncode}\n--- LOG TERAKHIR ---\n{tail}')
else: print('REUSE HASIL AF2RN LENGKAP:',RESULT)
if not RESULT.is_file(): raise FileNotFoundError(RESULT)
arm_result=json.loads(RESULT.read_text(encoding='utf-8'))
print('AF2RN METRICS:',{key:arm_result['metrics'][key] for key in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')})
print('TRAINING THIS CALL:',arm_result['training_executed_this_call']); print('TEST:',arm_result['test_images_accessed'])


In [ ]:
from coffee_detector.experiments.run_faruq_v3_af2rn import run_af2rn_seed42_decision
DECISION=OUTPUT/'val_reports/af2rn_seed42_decision.json'
decision=run_af2rn_seed42_decision(RESULT,AF2_RESULT,DECISION)
import pandas as pd
from IPython.display import display
rows=[{'model':model,**metrics} for model,metrics in decision['values'].items()]
display(pd.DataFrame(rows).style.format({key:'{:.2%}' for key in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')}))
print('DELTAS:',decision['deltas']); print('CRITERIA:',decision['criteria']); print('DECISION:',decision['decision']); print('NEXT:',decision['next']); print('TEST:',decision['test_opened'])


In [ ]:
output_manifest={'format':'coffee_detector.af2rn.kaggle_training_output.v1','branch':BRANCH,'commit':COMMIT,'gpu':GPU,'d0_checkpoint_sha256':CONTRACT['d0_checkpoint_sha256'],'af2_result_sha256':CONTRACT['af2_result_sha256'],'static_decision':static['decision'],'observability_decision':observability['decision'],'seed42_decision':decision['decision'],'training_executed':True,'test_images_accessed':False}
(OUTPUT/'output_manifest.json').write_text(json.dumps(output_manifest,indent=2)+'\n',encoding='utf-8')
archive=shutil.make_archive(str(WORK/'faruq-v3-af2rn-seed42-training-output'),'zip',root_dir=OUTPUT)
print('OUTPUT:',OUTPUT); print('BEST:',OUTPUT/'AF2RN/AF2RN_seed42/weights/best.pt'); print('LAST:',OUTPUT/'AF2RN/AF2RN_seed42/weights/last.pt'); print('DOWNLOAD ZIP:',archive)
print('Kirim tabel + DELTAS + DECISION. Jangan membuka test atau seed lain sebelum keputusan diperiksa.')
